# 04 - Interpretabilidad, Importancia Ponderada y Benchmark Integral de Ensambles

## Curso de Machine Learning: Modelos de Clasificación

En un modelo `AdaBoostClassifier`, la importancia global de cada característica se calcula como el **promedio ponderado** de las importancias de cada árbol individual, utilizando los pesos de autoridad $\alpha_m$ (`estimator_weights_`) como factores de ponderación:

$$I_j = \frac{\sum_{m=1}^M \alpha_m \cdot I_j^{(m)}}{\sum_{m=1}^M \alpha_m}$$

En este cuaderno inspeccionaremos los atributos internos aprendidos por AdaBoost (`estimators_`, `estimator_weights_`, `estimator_errors_`) y realizaremos un benchmark comparativo frente a los demás clasificadores del curso.

---

### Contenidos
1. Inspección Forense del Ensamble: Errores ($\varepsilon_m$) y Pesos ($\alpha_m$) por Árbol
2. Importancia de Características Ponderada vs Importancia por Permutación
3. Comparación de Fronteras de Decisión en 2D: Árbol Único vs AdaBoost vs Random Forest vs Gradient Boosting
4. Benchmark Integral de Clasificación en Dataset Clínico (`load_breast_cancer`)


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import DecisionBoundaryDisplay, permutation_importance
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


In [ ]:
# Entrenamiento e inspeccion de estimator_errors_ y estimator_weights_
cancer = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target)

ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)
ada_clf.fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(range(1, 101), ada_clf.estimator_errors_, color='#d62728', lw=1.8)
axes[0].set_title('Error Ponderado por Arbol (estimator_errors_)', fontweight='bold')
axes[0].set_xlabel('Indice del Estimador (m)')
axes[0].set_ylabel('Error Ponderado (epsilon_m)')

axes[1].plot(range(1, 101), ada_clf.estimator_weights_, color='#2b5c8f', lw=1.8)
axes[1].set_title('Autoridad / Peso de cada Arbol (estimator_weights_)', fontweight='bold')
axes[1].set_xlabel('Indice del Estimador (m)')
axes[1].set_ylabel('Peso del Modelo (alpha_m)')

plt.tight_layout()
plt.show()


In [ ]:
# Top 10 caracteristicas mas relevantes segun AdaBoost (feature_importances_)
importancias = ada_clf.feature_importances_
idx_top = np.argsort(importancias)[-10:]

plt.figure(figsize=(9, 4.5))
plt.barh(np.array(cancer.feature_names)[idx_top], importancias[idx_top], color='#d95f02')
plt.xlabel('Importancia Ponderada Acumulada en Decision Stumps', fontweight='bold')
plt.title('Top 10 Caracteristicas Seleccionadas por los Decision Stumps de AdaBoost', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Benchmark Integral de Ensambles y Clasificadores
modelos = {
    'Decision Stump (max_depth=1)': DecisionTreeClassifier(max_depth=1, random_state=42),
    'AdaBoost (100 Stumps)': AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42),
    'Random Forest (100 Trees)': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting (100 Trees)': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

tabla_bench = []
for nombre, m in modelos.items():
    t0 = time.perf_counter()
    m.fit(X_tr, y_tr)
    t_fit = (time.perf_counter() - t0) * 1000
    
    preds = m.predict(X_te)
    probs = m.predict_proba(X_te)[:, 1]
    tabla_bench.append({
        'Modelo': nombre,
        'Accuracy (%)': accuracy_score(y_te, preds) * 100,
        'F1-Score (%)': f1_score(y_te, preds) * 100,
        'ROC-AUC (%)': roc_auc_score(y_te, probs) * 100,
        'Tiempo Fit (ms)': t_fit
    })

df_b = pd.DataFrame(tabla_bench).sort_values(by='ROC-AUC (%)', ascending=False)
print(df_b.to_string(index=False))
